# Load the data for analysis 

In [1]:
import pandas as pd

# load the data to the 
df = pd.read_csv("../data/TXB_23_landingpage_clean.csv")
df.head()

,group_id,arm,visitor_id,arrival_time,exit_time,time_on_page_sec,exit_rate,ctr_opportunities,ctr_newsletter,ctr_donation,ctr_events,kpi_x,kpi_y,scroll_depth_pct,ctr_partner_banner,page_load_time_ms
0,TXB_23,treatment,355,2026-02-22 11:54:51.000000,2026-02-22 11:55:01.900000,10.9,0,0,1,0,0,13.14,0,86.4,0,2654.2
1,TXB_23,treatment,389,2026-02-22 22:03:14.000000,2026-02-22 22:07:29.400000,255.4,0,0,0,0,0,6.65,0,36.4,0,1451.9
2,TXB_23,treatment,781,2026-02-22 22:07:43.000000,2026-02-22 23:04:55.600000,3432.6,0,0,1,1,1,24.55,0,89.9,0,2517.1
3,TXB_23,pre,190,2026-02-22 22:22:10.000000,2026-02-22 22:22:52.000000,42.0,0,0,0,0,0,37.45,0,90.1,0,1192.1
4,TXB_23,pre,1,2026-02-22 23:31:43.201452,2026-02-22 23:33:13.784650,90.6,0,0,0,0,0,16.20,0,17.9,0,1966.6


# Get the observed counts 

In [ ]:
# get observed counts and total records 
observed_counts = df['arm'].value_counts().tolist()
total_users = sum(observed_counts)

print(f"Total Users: {total_users}")
print(f"Observed Split (Treatment, Control): {observed_counts}\n")

Total Users: 1163
Observed Split (Treatment, Control): [814, 349]



# Perform a Chi-Square statistical test

In [5]:
from scipy.stats import chisquare

# hypothesis 1: 50/50 intended split 
srm_50_50 = chisquare(f_obs=observed_counts)
print("50/50 Split:")
print(f"Chi-Square Statistic: {srm_50_50.statistic:.2f}")
print(f"P-value: {srm_50_50.pvalue:.2e}\n")

# hypothesis 2: 70/30 intended split
expected_70_30 = [total_users * 0.70, total_users * 0.30]
srm_70_30 = chisquare(f_obs=observed_counts, f_exp=expected_70_30)

print("70/30 Split:")
print(f"Chi-Square Statistic: {srm_70_30.statistic:.5f}")
print(f"P-value: {srm_70_30.pvalue:.3f}")

50/50 Split:
Chi-Square Statistic: 185.92
P-value: 2.47e-42

70/30 Split:
Chi-Square Statistic: 0.00004
P-value: 0.995


# Sample Ratio Mismatch (SRM) Analysis

**Objective:**
Before evaluating the causal impact of the proposed CTA button, we must verify the integrity of the experiment's traffic allocation. Because the intended traffic split was not explicitly provided prior to the analysis, we investigated the observed traffic under the two most likely experimental designs: a standard 50/50 split and an asymmetric 70/30 split.

**Observed Traffic:**
* Total Traffic: 1,163 users
* Treatment Group: 814 users (~70%)
* Control Group: 349 users (~30%)

**Hypothesis 1: Assuming a standard 50/50 intended split**
We utilized a Chi-Square Goodness-of-Fit test to compare our observed counts against an expected 50/50 distribution.
* **Chi-Square Statistic ($\chi^2$):** 185.92
* **p-value:** $2.47 \times 10^{-42}$
* **Conclusion:** If the intended split was 50/50, we have a severe Sample Ratio Mismatch (SRM). The randomization algorithm completely failed, meaning the fundamental assumption of causal inference is violated, and the data cannot be trusted.

**Hypothesis 2: Assuming an intentional 70/30 intended split**
Because the observed traffic falls almost perfectly into a 70% / 30% distribution, we ran a secondary Chi-Square test assuming this was the designed split.
* **Chi-Square Statistic ($\chi^2$):** 0.00004
* **p-value:** 0.995
* **Conclusion:** If the test was purposefully designed as a 70/30 split, there is no SRM. The randomization functioned flawlessly, and the data is mathematically safe for causal analysis.

**Final Recommendation:**
Without explicit confirmation of the intended split, we must proceed with caution. For the remainder of this analysis, we will assume the 70/30 split was intentional and proceed with evaluating the primary KPIs. However, if this was meant to be a 50/50 test, all downstream results should be discarded due to severe tracking bias.